# 台灣好聲音（PitchLab）雲端 Colab 分軌處理器
一鍵安裝 Demucs + yt-dlp，輸入本機音檔或 YouTube 網址，產出可直接匯入「台灣好聲音 PWA」的 `.zip` 歌包。

In [ ]:
!git clone https://github.com/bard0228/pitchlab.git /content/pitchlab 2>/dev/null || (cd /content/pitchlab && git pull)
!pip -q install -r /content/pitchlab/pipeline/requirements.txt
!apt-get -qq install -y ffmpeg

In [ ]:
#@title 選擇音檔或貼上 YouTube 網址（遇風控牆請上傳本機音檔）
YOUTUBE_URL = "" #@param {type:"string"}
SONG_TITLE = "" #@param {type:"string"}
ARTIST = "" #@param {type:"string"}
SAVE_TO_DRIVE = True #@param {type:"boolean"}

from google.colab import files
from pathlib import Path
import subprocess

out_dir = Path('/content/pitchlab_out')
out_dir.mkdir(exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = Path('/content/drive/MyDrive/PitchLabSongs')
    out_dir.mkdir(parents=True, exist_ok=True)

cmd = ['python3', '/content/pitchlab/pipeline/process_song.py', '--separator', 'demucs']
if YOUTUBE_URL.strip():
    cmd += ['--youtube', YOUTUBE_URL.strip()]
else:
    uploaded = files.upload()
    src_file = next(iter(uploaded.keys()))
    cmd += ['--input', src_file]

if SONG_TITLE.strip():
    cmd += ['--title', SONG_TITLE.strip()]
if ARTIST.strip():
    cmd += ['--artist', ARTIST.strip()]

out_name = (SONG_TITLE.strip() or 'song_package') + '.zip'
out_zip = out_dir / out_name
cmd += ['--out', str(out_zip)]

res = subprocess.run(cmd, check=False)
if res.returncode == 0:
    print('完成！歌包已儲存至：', out_zip)
    files.download(str(out_zip))